In [1]:
"""
================================================================================
  14_Classification_Baseline_Comparison.ipynb
  VGG-19 vs ResNet-50 vs EfficientNet-B4 vs DINOv2+Linear vs WILLIE
================================================================================

  PURPOSE: Justify WILLIE's classification performance against single-task
           baselines. Show that multi-task learning matches/exceeds specialized
           classifiers despite simultaneous seg + det training.

  PRODUCES (300 DPI, PNG + PDF):
    1. fig_cls_accuracy_bars       - Accuracy comparison all models
    2. fig_cls_grouped_metrics     - Acc/F1/AUC grouped bars
    3. fig_cls_fold_variance       - 5-fold CV variance per model
    4. fig_cls_efficiency          - Accuracy vs parameters bubble
    5. fig_cls_training_time       - Training time comparison
    6. fig_cls_auc_comparison      - AUC-ROC bar chart
    7. fig_cls_ranking_table       - Publication ranking table
    8. tbl_cls_full_comparison     - Complete results table
================================================================================
"""

import os, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

PROJECT_ROOT = "."
FIGURES_DIR = os.path.join(PROJECT_ROOT, "artifacts/14_cls_baseline_comparison/figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

plt.rcParams.update({
    'font.size': 11, 'axes.titlesize': 14, 'axes.labelsize': 12,
    'figure.dpi': 100, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'savefig.facecolor': 'white', 'axes.grid': True, 'grid.alpha': 0.3,
    'font.family': 'sans-serif',
})

def save_fig(fig, name, close=True):
    for ext in ['png', 'pdf']:
        fig.savefig(os.path.join(FIGURES_DIR, f"{name}.{ext}"),
                    dpi=300, bbox_inches='tight', facecolor='white')
    if close:
        plt.close(fig)
    print(f"  📈 {name} (.png + .pdf)")

print("=" * 80)
print("  Classification Baseline Comparison")
print("=" * 80)
print(f"  📁 Figures: {FIGURES_DIR}")

# ====================================================================
# RESULTS (from notebook 07 baselines + notebooks 09/10/11)
# ====================================================================

# Single-task baselines
BL = {
    'VGG-19':          {'acc': 78.63, 'f1': 78.14, 'auc': 0.956, 'params': 124.9,
                        'time_min': 6.9,
                        'folds': [74.79, 70.94, 73.93, 75.21, 77.35],
                        'type': 'baseline'},
    'EfficientNet-B4': {'acc': 83.76, 'f1': 83.36, 'auc': 0.975, 'params': 18.5,
                        'time_min': 9.7,
                        'folds': [85.47, 82.05, 82.91, 82.91, 82.48],
                        'type': 'baseline'},
    'DINOv2+Linear':   {'acc': 86.75, 'f1': 87.01, 'auc': 0.986, 'params': 22.2,
                        'time_min': 5.7,
                        'folds': [82.05, 85.04, 86.75, 87.18, 81.62],
                        'type': 'baseline'},
    'ResNet-50':       {'acc': 88.03, 'f1': 87.82, 'auc': 0.973, 'params': 24.6,
                        'time_min': 5.2,
                        'folds': [83.76, 79.49, 85.47, 84.62, 85.47],
                        'type': 'baseline'},
}

# WILLIE multi-task models
WS = {
    'WS-MINI':         {'acc': 86.80, 'f1': 85.40, 'auc': 0.968, 'params': 34.3,
                        'time_min': 480.0,
                        'folds': [86.1, 87.5, 84.3, 86.6, 87.5],
                        'type': 'willie'},
    'WS-BASE (TTA)':   {'acc': 91.88, 'f1': 91.14, 'auc': 0.992, 'params': 520.4,
                        'time_min': 2400.0,
                        'folds': [86.57, 88.0, 85.0, 90.0, 85.0],
                        'type': 'willie'},
    'WS-XL (TTA)':     {'acc': 91.88, 'f1': 90.73, 'auc': 0.986, 'params': 762.5,
                        'time_min': 4620.0,
                        'folds': [89.8, 91.2, 86.6, 86.3, 84.6],
                        'type': 'willie'},
}

ALL = {**BL, **WS}
ALL_NAMES = list(ALL.keys())

# Colors
COLORS_BL = ['#95a5a6', '#3498db', '#9b59b6', '#2ecc71']  # gray, blue, purple, green
COLORS_WS = ['#e67e22', '#e74c3c', '#c0392b']  # orange, red, dark red
COLORS_ALL = COLORS_BL + COLORS_WS

print(f"\n  {'Model':<20s} {'Acc':>7s} {'F1':>7s} {'AUC':>6s} {'Params':>8s}")
print(f"  {'_'*50}")
for name, m in ALL.items():
    print(f"  {name:<20s} {m['acc']:>6.2f}% {m['f1']:>6.2f}% {m['auc']:>5.3f} {m['params']:>7.1f}M")


# ====================================================================
# FIGURE 1: ACCURACY BARS (all models)
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 1: Accuracy Comparison")
print("=" * 70)

fig1, ax = plt.subplots(figsize=(12, 6))

accs = [ALL[n]['acc'] for n in ALL_NAMES]
bars = ax.bar(range(len(ALL_NAMES)), accs, color=COLORS_ALL,
              edgecolor='black', linewidth=0.5, alpha=0.85)

for bar, val in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Separator line between baselines and WILLIE
ax.axvline(3.5, color='black', linestyle='--', lw=1.5, alpha=0.4)
ax.text(1.5, 70, 'Single-Task\nBaselines', ha='center', fontsize=10,
        style='italic', alpha=0.6)
ax.text(5, 70, 'WoundSHoT\n(Multi-Task)', ha='center', fontsize=10,
        style='italic', alpha=0.6)

ax.set_xticks(range(len(ALL_NAMES)))
ax.set_xticklabels(ALL_NAMES, rotation=20, ha='right', fontweight='bold')
ax.set_ylabel('Test Accuracy (%)', fontweight='bold')
ax.set_title('Classification Accuracy: Single-Task Baselines vs WILLIE',
             fontweight='bold', fontsize=14, pad=15)
ax.set_ylim([65, 98])
save_fig(fig1, "fig_cls_accuracy_bars")


# ====================================================================
# FIGURE 2: GROUPED METRICS (Acc / F1 / AUC)
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 2: Grouped Metrics")
print("=" * 70)

fig2, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(ALL_NAMES))
w = 0.25

acc_vals = [ALL[n]['acc'] for n in ALL_NAMES]
f1_vals = [ALL[n]['f1'] for n in ALL_NAMES]
auc_vals = [ALL[n]['auc'] * 100 for n in ALL_NAMES]

b1 = ax.bar(x - w, acc_vals, w, label='Accuracy', color='#3498db', alpha=0.85)
b2 = ax.bar(x, f1_vals, w, label='F1 (macro)', color='#2ecc71', alpha=0.85)
b3 = ax.bar(x + w, auc_vals, w, label='AUC-ROC x100', color='#e74c3c', alpha=0.85)

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.2,
                    f'{h:.1f}', ha='center', va='bottom', fontsize=7)

ax.axvline(3.5, color='black', linestyle='--', lw=1.5, alpha=0.4)
ax.set_xticks(x)
ax.set_xticklabels(ALL_NAMES, rotation=20, ha='right', fontweight='bold')
ax.set_ylabel('Score (%)', fontweight='bold')
ax.set_title('Classification Metrics: Accuracy / F1 / AUC-ROC',
             fontweight='bold', fontsize=14, pad=15)
ax.legend(fontsize=10)
ax.set_ylim([65, 105])
save_fig(fig2, "fig_cls_grouped_metrics")


# ====================================================================
# FIGURE 3: 5-FOLD CV VARIANCE
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 3: Fold Variance")
print("=" * 70)

fig3, ax = plt.subplots(figsize=(12, 6))

names_with_folds = [n for n in ALL_NAMES if 'folds' in ALL[n]]
for i, name in enumerate(names_with_folds):
    folds = ALL[name]['folds']
    color = COLORS_ALL[ALL_NAMES.index(name)]
    ax.plot(range(5), folds, 'o-', color=color, lw=2, markersize=8,
            label=f"{name} ({np.mean(folds):.1f}% +/-{np.std(folds):.1f}%)")
    ax.axhline(np.mean(folds), color=color, linestyle=':', alpha=0.3)

ax.set_xticks(range(5))
ax.set_xticklabels([f'Fold {i}' for i in range(5)], fontweight='bold')
ax.set_ylabel('Validation Accuracy (%)', fontweight='bold')
ax.set_title('5-Fold Cross-Validation Stability',
             fontweight='bold', fontsize=14, pad=15)
ax.legend(fontsize=9, loc='lower left')
save_fig(fig3, "fig_cls_fold_variance")


# ====================================================================
# FIGURE 4: EFFICIENCY (Accuracy vs Parameters)
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 4: Efficiency")
print("=" * 70)

fig4, ax = plt.subplots(figsize=(10, 7))

for i, name in enumerate(ALL_NAMES):
    m = ALL[name]
    color = COLORS_ALL[i]
    marker = 's' if m['type'] == 'willie' else 'o'
    size = 200 if m['type'] == 'willie' else 150
    ax.scatter(m['params'], m['acc'], s=size, c=color,
               marker=marker, edgecolors='black', linewidths=1.5, zorder=3)
    offset_y = 1.0 if m['acc'] < 90 else -2.0
    ax.annotate(f"{name}\n{m['acc']:.1f}%",
                xy=(m['params'], m['acc']),
                xytext=(m['params'], m['acc'] + offset_y),
                ha='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Total Parameters (M)', fontweight='bold', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontweight='bold', fontsize=12)
ax.set_title('Classification Accuracy vs Model Size\n'
             '(circles = baselines, squares = WILLIE)',
             fontweight='bold', fontsize=14, pad=15)
ax.set_xscale('log')

# Annotate key finding
ax.text(0.02, 0.98,
        'WoundSHoT-BASE surpasses all baselines\n'
        'despite simultaneous seg + det training',
        transform=ax.transAxes, ha='left', va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#d5f5e3',
                  edgecolor='#27ae60', alpha=0.9))
save_fig(fig4, "fig_cls_efficiency")


# ====================================================================
# FIGURE 5: TRAINING TIME
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 5: Training Time")
print("=" * 70)

fig5, ax = plt.subplots(figsize=(10, 5))

# Only baselines (WILLIE times are for all 3 tasks, not comparable)
bl_names = list(BL.keys())
bl_times = [BL[n]['time_min'] for n in bl_names]
bl_colors = COLORS_BL

bars = ax.barh(range(len(bl_names)), bl_times, color=bl_colors,
               edgecolor='black', linewidth=0.5, alpha=0.85, height=0.6)
for bar, val in zip(bars, bl_times):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f} min', va='center', fontweight='bold', fontsize=11)

ax.set_yticks(range(len(bl_names)))
ax.set_yticklabels(bl_names, fontweight='bold')
ax.set_xlabel('Training Time (minutes)', fontweight='bold')
ax.set_title('Single-Task Baseline Training Time (5-fold CV)\n'
             'Total: ~30 min for all 4 models',
             fontweight='bold', fontsize=14, pad=15)
ax.set_xlim([0, 14])

# Total annotation
total_time = sum(bl_times)
ax.text(0.98, 0.05, f'Total: {total_time:.1f} min',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=11, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
save_fig(fig5, "fig_cls_training_time")


# ====================================================================
# FIGURE 6: AUC-ROC COMPARISON
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 6: AUC-ROC Comparison")
print("=" * 70)

fig6, ax = plt.subplots(figsize=(12, 5.5))

auc_vals = [ALL[n]['auc'] for n in ALL_NAMES]
bars = ax.bar(range(len(ALL_NAMES)), auc_vals, color=COLORS_ALL,
              edgecolor='black', linewidth=0.5, alpha=0.85)

for bar, val in zip(bars, auc_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.axvline(3.5, color='black', linestyle='--', lw=1.5, alpha=0.4)
ax.set_xticks(range(len(ALL_NAMES)))
ax.set_xticklabels(ALL_NAMES, rotation=20, ha='right', fontweight='bold')
ax.set_ylabel('AUC-ROC', fontweight='bold')
ax.set_title('Area Under ROC Curve (higher = better class separability)',
             fontweight='bold', fontsize=14, pad=15)
ax.set_ylim([0.93, 1.005])
save_fig(fig6, "fig_cls_auc_comparison")


# ====================================================================
# FIGURE 7: RANKING TABLE (visual)
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 7: Ranking Table")
print("=" * 70)

fig7, ax = plt.subplots(figsize=(12, 5))
ax.axis('off')

# Sort by accuracy
sorted_models = sorted(ALL.items(), key=lambda x: x[1]['acc'], reverse=True)

rank_data = [['Rank', 'Model', 'Type', 'Accuracy', 'F1', 'AUC', 'Params']]
for rank, (name, m) in enumerate(sorted_models, 1):
    mtype = 'Multi-Task' if m['type'] == 'willie' else 'Single-Task'
    rank_data.append([
        f'#{rank}', name, mtype,
        f"{m['acc']:.2f}%", f"{m['f1']:.2f}%",
        f"{m['auc']:.3f}", f"{m['params']:.1f}M"
    ])

table = ax.table(cellText=rank_data[1:], colLabels=rank_data[0],
                 cellLoc='center', loc='center',
                 colWidths=[0.06, 0.18, 0.12, 0.12, 0.12, 0.10, 0.10])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.7)

for j in range(7):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Color rows by type
for i in range(1, len(rank_data)):
    mtype = rank_data[i][2]
    bg = '#d5f5e3' if mtype == 'Multi-Task' else '#fef9e7'
    for j in range(7):
        table[i, j].set_facecolor(bg)

ax.set_title('Classification Model Ranking\n'
             'Green = WILLIE (multi-task), Yellow = single-task baseline',
             fontweight='bold', fontsize=13, pad=20)
save_fig(fig7, "fig_cls_ranking_table")


# ====================================================================
# FIGURE 8: FULL COMPARISON TABLE
# ====================================================================
print("\n" + "=" * 70)
print("  FIGURE 8: Full Comparison Table")
print("=" * 70)

fig8, ax = plt.subplots(figsize=(16, 6))
ax.axis('off')

tbl_data = [
    ['Model', 'Type', 'Params', 'Accuracy', 'F1 (macro)',
     'AUC-ROC', '5-Fold CV\nMean +/- Std', 'Train Time']
]
for name, m in ALL.items():
    mtype = 'Multi-Task' if m['type'] == 'willie' else 'Single-Task'
    folds = m.get('folds', [])
    if folds:
        cv_str = f'{np.mean(folds):.2f} +/- {np.std(folds):.2f}%'
    else:
        cv_str = '-'
    time_str = f'{m["time_min"]:.0f} min' if m['time_min'] < 60 else f'{m["time_min"]/60:.0f}h'
    tbl_data.append([
        name, mtype, f'{m["params"]:.1f}M',
        f'{m["acc"]:.2f}%', f'{m["f1"]:.2f}%',
        f'{m["auc"]:.3f}', cv_str, time_str
    ])

table = ax.table(cellText=tbl_data[1:], colLabels=tbl_data[0],
                 cellLoc='center', loc='center',
                 colWidths=[0.14, 0.10, 0.08, 0.10, 0.10, 0.09, 0.18, 0.09])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.7)

for j in range(8):
    table[0, j].set_facecolor('#2c3e50')
    table[0, j].set_text_props(color='white', fontweight='bold')

for i in range(1, len(tbl_data)):
    mtype = tbl_data[i][1]
    bg = '#d5f5e3' if mtype == 'Multi-Task' else 'white'
    for j in range(8):
        table[i, j].set_facecolor(bg)

ax.set_title('Complete Classification Comparison\n'
             'Single-Task Baselines vs WILLIE Multi-Task Models',
             fontweight='bold', fontsize=14, pad=25)
save_fig(fig8, "tbl_cls_full_comparison")


# ====================================================================
# SUMMARY
# ====================================================================

fig_files = sorted([f for f in os.listdir(FIGURES_DIR)
                    if f.endswith(('.png', '.pdf'))])

print(f"\n{'='*80}")
print(f"  CLASSIFICATION BASELINE COMPARISON COMPLETE")
print(f"{'='*80}")
print(f"""
  KEY FINDINGS:
  - Best single-task baseline: ResNet-50 at 88.03%
  - WILLIE-BASE (TTA) reaches 91.88% (+3.85% over best baseline)
  - WILLIE achieves this WHILE doing seg + det simultaneously
  - DINOv2+Linear has highest AUC (0.986) but lower accuracy (86.75%)
  - VGG-19 has 5x more params than ResNet-50 but 10% lower accuracy

  {FIGURES_DIR}
  {len(fig_files)} files generated
""")
for f in fig_files:
    size = os.path.getsize(os.path.join(FIGURES_DIR, f))
    icon = '📈' if f.endswith('.png') else '📄'
    print(f"     {icon} {f}  ({size/1024:.1f} KB)")

  Classification Baseline Comparison
  📁 Figures: artifacts/14_cls_baseline_comparison/figures

  Model                    Acc      F1    AUC   Params
  __________________________________________________
  VGG-19                78.63%  78.14% 0.956   124.9M
  EfficientNet-B4       83.76%  83.36% 0.975    18.5M
  DINOv2+Linear         86.75%  87.01% 0.986    22.2M
  ResNet-50             88.03%  87.82% 0.973    24.6M
  WS-MINI               86.80%  85.40% 0.968    34.3M
  WS-BASE (TTA)         91.88%  91.14% 0.992   520.4M
  WS-XL (TTA)           91.88%  90.73% 0.986   762.5M

  FIGURE 1: Accuracy Comparison
  📈 fig_cls_accuracy_bars (.png + .pdf)

  FIGURE 2: Grouped Metrics
  📈 fig_cls_grouped_metrics (.png + .pdf)

  FIGURE 3: Fold Variance
  📈 fig_cls_fold_variance (.png + .pdf)

  FIGURE 4: Efficiency
  📈 fig_cls_efficiency (.png + .pdf)

  FIGURE 5: Training Time
  📈 fig_cls_training_time (.png + .pdf)

  FIGURE 6: AUC-ROC Comparison
  📈 fig_cls_auc_comparison (.png + .pdf)

  FIG